# Aula 01 — Ranking das Escolas do RN (ENEM 2025) — versão Colab

Adaptação do `main.py` do AVA (Exemplo Prático da Aula 01). Mesma lógica; só muda caminho base e a saída (download em vez de `webbrowser`).

**Arquivos necessários (baixar do AVA):**
- `RESULTADOS_2025_RN.csv` (Microdados ENEM 2025 RN)
- `catalogo_escolas_rn.csv` (Catálogo Escola RN)
- `Exemplo Análise Escolas.zip` (contém `template_escolas.html`)

In [ ]:
# === SETUP COLAB ===
# Envie os 3 arquivos quando o seletor abrir.
# Alternativa sem reenvio: monte o Drive e ajuste BASE_DIR
#   from google.colab import drive; drive.mount("/content/drive")
#   BASE_DIR = Path("/content/drive/MyDrive/ifrn/unidade1/aula01")
import os, shutil, zipfile
from pathlib import Path

try:
    from google.colab import files
    enviados = files.upload()
except ImportError:
    enviados = {}

os.makedirs("microdados", exist_ok=True)
for nome in enviados:
    if nome.lower().endswith(".zip"):
        with zipfile.ZipFile(nome) as z:
            for m in z.namelist():
                if m.endswith("template_escolas.html") and not m.startswith("__MACOSX"):
                    with open("template_escolas.html", "wb") as f:
                        f.write(z.read(m))
    elif nome.lower().endswith(".csv"):
        destino = "RESULTADOS_2025_RN.csv" if "RESULTADO" in nome.upper() or "MICRODADOS" in nome.upper() else "catalogo_escolas_rn.csv"
        shutil.move(nome, f"microdados/{destino}")

print(os.listdir("microdados"), os.path.exists("template_escolas.html"))

In [ ]:
import csv
import json
from statistics import mean

BASE_DIR = Path(".")
RESULTADOS_CSV = BASE_DIR / "microdados" / "RESULTADOS_2025_RN.csv"
CATALOGO_CSV = BASE_DIR / "microdados" / "catalogo_escolas_rn.csv"
OUTPUT_HTML = BASE_DIR / "escolas_rn.html"

TIPO_DEPENDENCIA = {"1": "Federal", "2": "Estadual", "3": "Municipal", "4": "Privada"}

NOTA_COLUNAS = {
    "NU_NOTA_CN": "cn",
    "NU_NOTA_CH": "ch",
    "NU_NOTA_LC": "lc",
    "NU_NOTA_MT": "mt",
    "NU_NOTA_REDACAO": "redacao",
}

In [ ]:
def carregar_catalogo_nomes():
    nomes = {}
    with open(CATALOGO_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            nomes[row["CO_ESCOLA"]] = row["NO_ESCOLA"]
    return nomes


def carregar_escolas():
    nomes = carregar_catalogo_nomes()
    escolas = {}

    with open(RESULTADOS_CSV, encoding="latin1") as f:
        for row in csv.DictReader(f, delimiter=";"):
            codigo = row["CO_ESCOLA"]
            if not codigo:
                continue

            escola = escolas.setdefault(codigo, {
                "codigo": codigo,
                "municipio": row["NO_MUNICIPIO_ESC"],
                "tipo": TIPO_DEPENDENCIA.get(row["TP_DEPENDENCIA_ADM_ESC"], "-"),
                "alunos": [],
            })

            notas_aluno = {}
            for coluna, chave in NOTA_COLUNAS.items():
                valor = row[coluna]
                notas_aluno[chave] = round(float(valor), 1) if valor else None

            validas = [v for v in notas_aluno.values() if v is not None]
            notas_aluno["geral"] = round(mean(validas), 1) if validas else None
            escola["alunos"].append(notas_aluno)

    lista = []
    for escola in escolas.values():
        # aluno sem nenhuma nota valida vai para o final
        alunos_ordenados = sorted(
            escola["alunos"],
            key=lambda a: a["geral"] if a["geral"] is not None else -1,
            reverse=True,
        )
        lista.append({
            "codigo": escola["codigo"],
            "nome": nomes.get(escola["codigo"], ""),
            "municipio": escola["municipio"],
            "tipo": escola["tipo"],
            "alunos": alunos_ordenados,
        })
    return lista

In [ ]:
def gerar_html(escolas):
    template = (BASE_DIR / "template_escolas.html").read_text(encoding="utf-8")
    html = template.replace("__DADOS_ESCOLAS__", json.dumps(escolas, ensure_ascii=False))
    OUTPUT_HTML.write_text(html, encoding="utf-8")


escolas = carregar_escolas()
gerar_html(escolas)
print(f"{len(escolas)} escolas processadas.")
print(f"Arquivo gerado: {OUTPUT_HTML}")

try:
    files.download(str(OUTPUT_HTML))   # substitui webbrowser.open
except NameError:
    pass

In [ ]:
# Exploração extra (opcional): top 10 escolas por média geral dos alunos com nota
resumo = []
for e in escolas:
    gerais = [a["geral"] for a in e["alunos"] if a["geral"] is not None]
    if gerais:
        resumo.append((round(mean(gerais), 1), len(gerais), e["nome"] or e["codigo"], e["tipo"], e["municipio"]))

for media, n, nome, tipo, mun in sorted(resumo, reverse=True)[:10]:
    print(f"{media:6.1f} | n={n:4d} | {tipo:<9} | {mun:<20} | {nome}")

## Discussão (fases 5 e 6 do CRISP-DM)

1. Médias de escolas com poucos participantes são comparáveis às de escolas grandes? (veja a coluna `n` acima)
2. O que o recorte "melhores N alunos" muda no ranking?
3. Que cuidados éticos um ranking de escolas exige?

_Suas anotações:_